In [1]:
import os
import random
import pickle
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy.io
from scipy.io import loadmat

import mne
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from tensorflow.keras.constraints import max_norm
from tensorflow.keras.layers import (
    Activation,
    AveragePooling2D,
    BatchNormalization,
    Conv2D,
    Dense,
    DepthwiseConv2D,
    Dropout,
    Flatten,
    Input,
    Reshape,
    SeparableConv2D,
    SpatialDropout2D,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

2026-04-24 03:18:20.811655: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777000701.014590      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777000701.076329      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777000701.551923      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777000701.551970      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777000701.551973      23 computation_placer.cc:177] computation placer alr

In [2]:
class EEGDataset_ADHD_TF:
    """
    Subject-level dataset (one .mat file = one subject)
    Each item: (fname, eeg_epochs_np, label_int)
      eeg_epochs_np: (E, C, T)
    """

    def __init__(self, adhd_dir, control_dir,
                 lowcut=0.5, highcut=60.0, notch=50.0,
                 window=2.0, overlap=0.5,
                 default_fs=128):

        self.samples = []
        self.lowcut = float(lowcut)
        self.highcut = float(highcut)
        self.notch = float(notch)
        self.window = float(window)
        self.overlap = float(overlap)
        self.default_fs = int(default_fs)

        self._process_folder(adhd_dir, label=1)
        self._process_folder(control_dir, label=0)

        if len(self.samples) == 0:
            raise ValueError("No subjects loaded. Check folders and file contents.")

    def _process_folder(self, folder, label):
        if not os.path.isdir(folder):
            raise ValueError(f"Directory does not exist: {folder}")

        for fname in sorted(os.listdir(folder)):
            if not fname.lower().endswith(".mat"):
                continue
            mat_path = os.path.join(folder, fname)
            eeg = self._process_mat(mat_path)
            if eeg is not None:
                self.samples.append((fname, eeg.astype(np.float32), int(label)))

    def _process_mat(self, file_path):
        mat = loadmat(file_path)

        # Variable name is usually the same as filename (e.g., v10p.mat -> "v10p")
        key = os.path.splitext(os.path.basename(file_path))[0]
        if key not in mat:
            # fallback: first 2D array
            key = None
            for k, v in mat.items():
                if isinstance(v, np.ndarray) and v.ndim == 2 and v.size > 0:
                    key = k
                    break
            if key is None:
                return None

        data = np.asarray(mat[key], dtype=np.float64)  # often (T, C)

        # Find fs if present
        fs = None
        for k in mat.keys():
            kl = k.lower()
            if ("fs" in kl) or ("freq" in kl) or ("sampling" in kl) or ("sfreq" in kl):
                try:
                    fs = int(np.squeeze(mat[k]))
                    break
                except Exception:
                    fs = None
        if fs is None or fs <= 0:
            fs = self.default_fs

        # Ensure (C, T) for MNE
        if data.ndim != 2:
            return None

        if data.shape[1] <= 256 and data.shape[0] > data.shape[1]:
            data = data.T  # (C, T)

        n_ch, n_times = data.shape
        min_samples = int(np.ceil(self.window * fs))
        if n_times < min_samples:
            return None

        ch_names = [f"Ch{i+1}" for i in range(n_ch)]
        info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=["eeg"] * n_ch)
        raw = mne.io.RawArray(data, info, verbose=False)

        raw.set_eeg_reference("average", verbose=False)
        raw.notch_filter(freqs=[self.notch], method="iir", verbose=False)
        raw.filter(self.lowcut, self.highcut, method="iir", verbose=False)

        step = self.window * (1.0 - self.overlap)
        if step <= 0:
            raise ValueError("overlap must be < 1.0 so that step > 0")

        epochs = mne.make_fixed_length_epochs(
            raw,
            duration=self.window,
            overlap=self.window - step,
            preload=True,
            verbose=False
        )

        eeg_data = epochs.get_data()  # (E, C, T)
        if eeg_data.size == 0:
            return None

        return eeg_data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [3]:
# Build epoch-level arrays from subject-level dataset
def build_epoch_arrays(dataset_obj):
    X_list, y_list, g_list = [], [], []

    for i in range(len(dataset_obj)):
        name, eeg_epochs, label = dataset_obj[i]  # (E,C,T)
        E = int(eeg_epochs.shape[0])

        X_list.append(eeg_epochs)
        y_list.append(np.full((E,), int(label), dtype=np.int32))
        g_list.append(np.full((E,), str(name), dtype=object))

    X = np.concatenate(X_list, axis=0).astype(np.float32)   # (N,C,T)
    y = np.concatenate(y_list, axis=0).astype(np.int32)     # (N,)
    groups = np.concatenate(g_list, axis=0)                 # (N,) subject-name per epoch

    return X, y, groups


# Subject-level labels derived from epoch-level groups/y
def build_subject_table_from_epochs(groups_epoch, y_epoch):
    subj_names = np.unique(groups_epoch.astype(str))
    subj_labels = []

    for s in subj_names:
        ys = np.unique(y_epoch[groups_epoch.astype(str) == s])
        if len(ys) != 1:
            raise ValueError(f"Subject {s} has multiple labels across epochs: {ys}")
        subj_labels.append(int(ys[0]))

    return subj_names, np.array(subj_labels, dtype=np.int32)


def epoch_indices_from_subjects(groups_epoch, subject_names_subset):
    subject_names_subset = np.array(subject_names_subset).astype(str)
    return np.where(np.isin(groups_epoch.astype(str), subject_names_subset))[0]


# tf.data builders
def make_ds_from_indices(X, y, groups, idxs, training, with_groups, batch_size, seed):
    x = X[idxs]
    yy = y[idxs]

    if with_groups:
        gg = groups[idxs].astype(str)
        ds = tf.data.Dataset.from_tensor_slices((x, yy, gg))
    else:
        ds = tf.data.Dataset.from_tensor_slices((x, yy))

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Validation Balanced Accuracy callback (epoch-level)
class ValBalancedAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, val_ds_xy):
        super().__init__()
        self.val_ds = val_ds_xy
        self.best = -np.inf
        self.last = None

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for xb, yb in self.val_ds:
            prob = self.model(xb, training=False).numpy().reshape(-1)
            pred = (prob >= 0.5).astype(int)
            y_true.extend(yb.numpy().tolist())
            y_pred.extend(pred.tolist())

        bacc = balanced_accuracy_score(y_true, y_pred)
        self.last = float(bacc)
        self.best = max(self.best, self.last)
        print(f" - val_balanced_accuracy: {self.last:.4f}", end="")

# Plotting functions
def plot_fold_training_curve(history, fold_id, save_path, metrics=("loss", "acc", "auc")):
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes = [axes]
    fig.suptitle(f"Fold {fold_id + 1} — Training Curves (Final Retrain)", fontsize=13)
    epochs_range = range(1, len(history["loss"]) + 1)
    for col, m in enumerate(metrics):
        ax = axes[col]
        if m in history:
            ax.plot(epochs_range, history[m], label=f"train_{m}", linewidth=1.2)
        val_key = f"val_{m}"
        if val_key in history:
            ax.plot(epochs_range, history[val_key], label=val_key, linewidth=1.2, linestyle="--")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")

# Fold-level training curves for all folds in a grid + overlay of all folds per metric
def plot_all_training_curves(fold_histories, save_dir, metrics=("loss", "acc", "auc")):
    n_folds = len(fold_histories)
    n_metrics = len(metrics)

    fig, axes = plt.subplots(n_folds, n_metrics, figsize=(5 * n_metrics, 3.5 * n_folds), squeeze=False)
    fig.suptitle("Training Curves per Fold (Final Retrain)", fontsize=14, y=1.01)
    for row, fid in enumerate(sorted(fold_histories.keys())):
        hist = fold_histories[fid]
        epochs_range = range(1, len(hist["loss"]) + 1)
        for col, m in enumerate(metrics):
            ax = axes[row, col]
            if m in hist:
                ax.plot(epochs_range, hist[m], label=f"train_{m}", linewidth=1.0)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=val_key, linewidth=1.0, linestyle="--")
            ax.set_title(f"Fold {fid + 1} — {m}", fontsize=9)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(m)
            ax.legend(fontsize=6)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, "training_curves_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, axes2 = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes2 = [axes2]
    fig2.suptitle("Training Curves Overlay (All Folds)", fontsize=14)
    for col, m in enumerate(metrics):
        ax = axes2[col]
        for fid in sorted(fold_histories.keys()):
            hist = fold_histories[fid]
            epochs_range = range(1, len(hist["loss"]) + 1)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
            elif m in hist:
                ax.plot(epochs_range, hist[m], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
        ax.set_title(f"All Folds — {m}", fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "training_curves_overlay.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_fold_confusion_matrix(cm, fold_id, save_path, class_names=("Control", "ADHD")):
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Fold {fold_id + 1} — Confusion Matrix (Subject-Level)", fontsize=11)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")

# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_all_confusion_matrices(fold_cms, super_cm, save_dir, class_names=("Control", "ADHD")):
    n_folds = len(fold_cms)
    ncols = min(n_folds, 5)
    nrows = int(np.ceil(n_folds / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    fig.suptitle("Confusion Matrices per Fold (Subject-Level)", fontsize=14, y=1.01)
    for i, fid in enumerate(sorted(fold_cms.keys())):
        row, col = divmod(i, ncols)
        ax = axes[row][col]
        disp = ConfusionMatrixDisplay(fold_cms[fid], display_labels=class_names)
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(f"Fold {fid + 1}", fontsize=10)
    for j in range(i + 1, nrows * ncols):
        row, col = divmod(j, ncols)
        axes[row][col].axis("off")
    plt.tight_layout()
    path = os.path.join(save_dir, "confusion_matrices_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, ax2 = plt.subplots(1, 1, figsize=(5, 5))
    disp2 = ConfusionMatrixDisplay(super_cm, display_labels=class_names)
    disp2.plot(ax=ax2, cmap="Oranges", colorbar=False, values_format="d")
    ax2.set_title("Accumulated Confusion Matrix (All Folds)", fontsize=12)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "confusion_matrix_accumulated.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

In [5]:
def EEGNet(nb_classes, Chans = 64, Samples = 128, 
             dropoutRate = 0.5, kernLength = 64, F1 = 8, 
             D = 2, F2 = 16, norm_rate = 0.25, dropoutType = 'Dropout'):

    if dropoutType == 'SpatialDropout2D':
        dropoutType = SpatialDropout2D
    elif dropoutType == 'Dropout':
        dropoutType = Dropout
    else:
        raise ValueError('dropoutType must be one of SpatialDropout2D '
                         'or Dropout, passed as a string.')

    input1 = Input(shape=(Chans, Samples))
    input1 = Reshape((Chans, Samples, 1))(input1)

    block1 = Conv2D(F1, (1, kernLength), padding='same',
                    name='Conv2D_1',
                    input_shape=(Chans, Samples, 1),
                    use_bias=False)(input1)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D((Chans, 1), use_bias=False,
                             name='Depth_wise_Conv2D_1',
                             depth_multiplier=D,
                             depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = dropoutType(dropoutRate)(block1)

    block2 = SeparableConv2D(F2, (1, 16),
                             name='Separable_Conv2D_1',
                             use_bias=False, padding='same')(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = dropoutType(dropoutRate)(block2)

    flatten = Flatten(name='flatten')(block2)

    dense = Dense(nb_classes, name='output',
                  kernel_constraint=max_norm(norm_rate))(flatten)
    softmax = Activation('softmax', name='out_activation')(dense)

    return Model(inputs=input1, outputs=softmax)

In [ ]:
# ============================================================
# SEED = 123
# ============================================================
SEED = 123
MODEL_NAME = "eegnet"

N_FOLDS = 5
BATCH_SIZE = 64
EPOCHS_FINAL = 100

RESULTS_ROOT = "/kaggle/working/results_eegnet_fixed_seed123"
CURVES_DIR   = os.path.join(RESULTS_ROOT, "training_curves")
MODELS_DIR   = os.path.join(RESULTS_ROOT, "models")
CM_DIR       = os.path.join(RESULTS_ROOT, "cm")
WEIGHTS_DIR  = os.path.join(RESULTS_ROOT, "weights")

for d in [RESULTS_ROOT, CURVES_DIR, MODELS_DIR, CM_DIR, WEIGHTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ------------------------------------------------------------
# Seed helper
# ------------------------------------------------------------
def set_seed(seed=None, deterministic=True):
    if seed is None:
        seed = SEED

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    if deterministic:
        os.environ["TF_DETERMINISTIC_OPS"] = "1"
        try:
            tf.config.experimental.enable_op_determinism()
        except Exception:
            pass

set_seed(SEED)

EXCLUDED_SUBJECTS = {"v56p"}

# ============================================================
# HPs FIJOS ORIGINALES DE EEGNET
# ============================================================
EEGNET_FIXED_MODEL_ARGS = {
    "nb_classes": 2,
    "dropoutRate": 0.5,
    "kernLength": 64,
    "F1": 8,
    "D": 2,
    "F2": 16,
    "norm_rate": 0.25,
    "dropoutType": "Dropout",
}

EEGNET_FIXED_COMPILE_ARGS = {
    "learning_rate": 1e-3,
}

# ============================================================
# DATASET BUILDER PARA EEGNET (softmax -> y one-hot)
# ============================================================
def make_eegnet_ds_from_indices(X, y, idxs, training, batch_size, seed):
    x = X[idxs].astype(np.float32)
    y_bin = y[idxs].astype(np.int32)
    y_oh = tf.keras.utils.to_categorical(y_bin, num_classes=2).astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((x, y_oh))

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# ============================================================
# EXTRAER PROBABILIDAD POSITIVA DESDE SOFTMAX
# ============================================================
def extract_positive_prob_softmax(model_output):
    out = model_output
    if tf.is_tensor(out):
        out = out.numpy()

    out = np.asarray(out)

    if out.ndim == 2 and out.shape[1] == 2:
        return out[:, 1].astype(np.float32)

    raise ValueError(f"Se esperaba salida softmax (B,2), pero llegó {out.shape}")

# ============================================================
# DATA
# ============================================================
root = "/kaggle/input/datasets/javierceron/tdha-data"
adhd_dir = os.path.join(root, "ADHD")
control_dir = os.path.join(root, "Control")

dataset_tf = EEGDataset_ADHD_TF(
    adhd_dir=adhd_dir,
    control_dir=control_dir,
    lowcut=0.5,
    highcut=60.0,
    notch=50.0,
    window=2.0,
    overlap=0.5,
    default_fs=128
)

# excluir sujetos problemáticos
n_before = len(dataset_tf.samples)
dataset_tf.samples = [
    sample for sample in dataset_tf.samples
    if os.path.splitext(sample[0])[0] not in EXCLUDED_SUBJECTS
]
n_after = len(dataset_tf.samples)

removed = n_before - n_after
print(f"Excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
print(f"Subjects removed: {removed}")
print(f"Remaining subjects: {n_after}")

if len(dataset_tf.samples) == 0:
    raise ValueError("No subjects left after exclusion. Check EXCLUDED_SUBJECTS.")

X, y, groups = build_epoch_arrays(dataset_tf)

print("Epoch arrays:")
print("  X:", X.shape, X.dtype)
print("  y:", y.shape, "labels:", np.unique(y))
print("  groups:", groups.shape, "n_subjects:", len(np.unique(groups.astype(str))))

N_CH, N_T = int(X.shape[1]), int(X.shape[2])
print("Model input shape (C,T):", (N_CH, N_T))

subj_names, subj_labels = build_subject_table_from_epochs(groups, y)
print("Subject table:")
print("  n_subjects:", len(subj_names))
print("  Control:", int(np.sum(subj_labels == 0)), "| ADHD:", int(np.sum(subj_labels == 1)))

# ============================================================
# OUTER CV: SUBJECT-INDEPENDENT
# ============================================================
outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
X_dummy = np.zeros((len(subj_names), 1), dtype=np.float32)

final_results = []
super_cm = np.zeros((2, 2), dtype=int)
fold_histories = {}
fold_cms = {}

for fold_id, (trainval_subj_idx, test_subj_idx) in enumerate(
    outer.split(X_dummy, subj_labels, groups=subj_names)
):
    print(f"\n================ Fold {fold_id+1}/{N_FOLDS} ================")

    inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + fold_id)

    inner_X_dummy = np.zeros((len(trainval_subj_idx), 1), dtype=np.float32)
    inner_y = subj_labels[trainval_subj_idx]
    inner_groups = subj_names[trainval_subj_idx]

    tr_rel, val_rel = next(inner.split(inner_X_dummy, inner_y, groups=inner_groups))

    train_subj_idx = trainval_subj_idx[tr_rel]
    val_subj_idx   = trainval_subj_idx[val_rel]

    tr_names = subj_names[train_subj_idx]
    va_names = subj_names[val_subj_idx]
    te_names = subj_names[test_subj_idx]

    tr_labels = subj_labels[train_subj_idx]
    va_labels = subj_labels[val_subj_idx]
    te_labels = subj_labels[test_subj_idx]

    if len(np.unique(tr_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: train set does not contain both classes.")
    if len(np.unique(va_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: validation set does not contain both classes.")
    if len(np.unique(te_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: test set does not contain both classes.")

    tr_idx  = epoch_indices_from_subjects(groups, tr_names)
    val_idx = epoch_indices_from_subjects(groups, va_names)
    te_idx  = epoch_indices_from_subjects(groups, te_names)

    trainval_names = np.concatenate([tr_names, va_names])
    trainval_idx = epoch_indices_from_subjects(groups, trainval_names)

    trainval_ds_eegnet = make_eegnet_ds_from_indices(
        X, y, trainval_idx,
        training=True,
        batch_size=BATCH_SIZE,
        seed=SEED
    )

    test_ds_xyg = make_ds_from_indices(
        X, y, groups, te_idx,
        training=False,
        with_groups=True,
        batch_size=BATCH_SIZE,
        seed=SEED
    )

    tf.keras.backend.clear_session()
    set_seed(SEED)

    model_args = dict(EEGNET_FIXED_MODEL_ARGS)
    model_args["Chans"] = N_CH
    model_args["Samples"] = N_T

    model = EEGNet(
        nb_classes=model_args["nb_classes"],
        Chans=model_args["Chans"],
        Samples=model_args["Samples"],
        dropoutRate=model_args["dropoutRate"],
        kernLength=model_args["kernLength"],
        F1=model_args["F1"],
        D=model_args["D"],
        F2=model_args["F2"],
        norm_rate=model_args["norm_rate"],
        dropoutType=model_args["dropoutType"],
    )

    opt = Adam(learning_rate=EEGNET_FIXED_COMPILE_ARGS["learning_rate"])

    model.compile(
        optimizer=opt,
        loss="categorical_crossentropy",
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="acc"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

    hist = model.fit(
        trainval_ds_eegnet,
        epochs=EPOCHS_FINAL,
        verbose=1,
    )

    fold_histories[fold_id] = hist.history

    model.save(os.path.join(MODELS_DIR, f"fold_{fold_id}.keras"))
    model.save_weights(os.path.join(WEIGHTS_DIR, f"fold_{fold_id}.weights.h5"))
    print(f"Fold {fold_id} model + weights saved. Trained {len(hist.history['loss'])} epochs.")

    plot_fold_training_curve(
        hist.history,
        fold_id,
        save_path=os.path.join(CURVES_DIR, f"fold_{fold_id}_curves.png"),
        metrics=("loss", "acc", "auc"),
    )

    # ========================================================
    # TEST EVALUATION A NIVEL DE SUJETO
    # ========================================================
    subject_preds = defaultdict(list)
    subject_probs = defaultdict(list)
    subject_true  = {}

    for xb, yb, sb in test_ds_xyg:
        raw_pred = model(xb, training=False)
        prob = extract_positive_prob_softmax(raw_pred)
        pred = (prob >= 0.5).astype(int)

        y_np = yb.numpy().astype(int)
        s_np = sb.numpy().astype(str)

        for name, p, pr, yt in zip(s_np, pred, prob, y_np):
            subject_preds[name].append(int(p))
            subject_probs[name].append(float(pr))
            subject_true[name] = int(yt)

    y_true, y_pred, y_prob = [], [], []
    for subj in sorted(subject_preds.keys()):
        y_true.append(subject_true[subj])
        y_pred.append(int(np.bincount(subject_preds[subj]).argmax()))
        y_prob.append(float(np.mean(subject_probs[subj])))

    acc  = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)

    if len(np.unique(y_true)) < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(y_true, y_prob)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    super_cm += cm
    fold_cms[fold_id] = cm

    plot_fold_confusion_matrix(
        cm,
        fold_id,
        save_path=os.path.join(CM_DIR, f"fold_{fold_id}_cm.png"),
    )

    final_results.append({
        "fold": fold_id,
        "accuracy": float(acc),
        "balanced_acc": float(bacc),
        "f1": float(f1),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "model_args": dict(model_args),
        "compile_args": dict(EEGNET_FIXED_COMPILE_ARGS),
    })

    print(f"Fold {fold_id} | Acc={acc:.3f} | BAcc={bacc:.3f} | F1={f1:.3f} | AUC={auc:.3f}")
    print("Confusion matrix:\n", cm)

# ============================================================
# REPORTES FINALES
# ============================================================
plot_all_training_curves(
    fold_histories,
    CURVES_DIR,
    metrics=("loss", "acc", "auc"),
)
plot_all_confusion_matrices(fold_cms, super_cm, CM_DIR)

print("\nACCUMULATED CONFUSION MATRIX")
print(super_cm)

df_res = pd.DataFrame(final_results)

print("\nFINAL RESULTS")
print(df_res[["accuracy", "balanced_acc", "f1", "auc"]].agg(["mean", "std"]))

df_res.to_csv(os.path.join(RESULTS_ROOT, "final_results.csv"), index=False)
print("Saved:", os.path.join(RESULTS_ROOT, "final_results.csv"))

Excluded subjects: ['v56p']
Subjects removed: 1
Remaining subjects: 120
Epoch arrays:
  X: (16640, 19, 256) float32
  y: (16640,) labels: [0 1]
  groups: (16640,) n_subjects: 120
Model input shape (C,T): (19, 256)
Subject table:
  n_subjects: 120
  Control: 59 | ADHD: 61

================ Fold 1/5 ================


I0000 00:00:1777000743.951045      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100


E0000 00:00:1777000750.271578      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1777000750.582659      67 cuda_dnn.cc:529] Loaded cuDNN version 91002


204/204 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - acc: 0.5574 - auc: 0.5723 - loss: 0.6905
Epoch 2/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.6750 - auc: 0.7296 - loss: 0.6103
Epoch 3/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7468 - auc: 0.8176 - loss: 0.5278
Epoch 4/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7945 - auc: 0.8669 - loss: 0.4604
Epoch 5/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8126 - auc: 0.8939 - loss: 0.4127
Epoch 6/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8346 - auc: 0.9106 - loss: 0.3800
Epoch 7/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8524 - auc: 0.9254 - loss: 0.3507
Epoch 8/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8473 - auc: 0.9273 - loss: 0.3439
Epoch 9/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8528 - auc: 0.9306 - loss: 0.3359
Epoch 10/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8707 - auc: 0.9434 - loss: 0.3061
Epoch 11/100
204/204 ━━━━━━━━━━━━━━━━━━━━

...

214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - acc: 0.5501 - auc: 0.5616 - loss: 0.6908
Epoch 2/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.6702 - auc: 0.7258 - loss: 0.6144
Epoch 3/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7465 - auc: 0.8173 - loss: 0.5298
Epoch 4/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7809 - auc: 0.8543 - loss: 0.4801
Epoch 5/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7991 - auc: 0.8756 - loss: 0.4475
Epoch 6/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8242 - auc: 0.9041 - loss: 0.3961
Epoch 7/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8426 - auc: 0.9155 - loss: 0.3756
Epoch 8/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8519 - auc: 0.9284 - loss: 0.3443
Epoch 9/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8569 - auc: 0.9340 - loss: 0.3296
Epoch 10/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8611 - auc: 0.9368 - loss: 0.3217
Epoch 11/100
214/214 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1777001464.000864      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


210/210 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - acc: 0.5494 - auc: 0.5682 - loss: 0.6878
Epoch 2/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.6968 - auc: 0.7621 - loss: 0.5874
Epoch 3/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7763 - auc: 0.8534 - loss: 0.4822
Epoch 4/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8229 - auc: 0.8958 - loss: 0.4148
Epoch 5/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8376 - auc: 0.9128 - loss: 0.3790
Epoch 6/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8391 - auc: 0.9177 - loss: 0.3684
Epoch 7/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8601 - auc: 0.9342 - loss: 0.3312
Epoch 8/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8645 - auc: 0.9406 - loss: 0.3169
Epoch 9/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8745 - auc: 0.9435 - loss: 0.3070
Epoch 10/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8807 - auc: 0.9486 - loss: 0.2927
Epoch 11/100
210/210 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1777001708.752925      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - acc: 0.5635 - auc: 0.5777 - loss: 0.6881
Epoch 2/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.6946 - auc: 0.7521 - loss: 0.5965
Epoch 3/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7508 - auc: 0.8207 - loss: 0.5243
Epoch 4/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7765 - auc: 0.8518 - loss: 0.4825
Epoch 5/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8054 - auc: 0.8813 - loss: 0.4358
Epoch 6/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8205 - auc: 0.8981 - loss: 0.4059
Epoch 7/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8345 - auc: 0.9093 - loss: 0.3832
Epoch 8/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8424 - auc: 0.9221 - loss: 0.3582
Epoch 9/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8554 - auc: 0.9315 - loss: 0.3361
Epoch 10/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8569 - auc: 0.9333 - loss: 0.3319
Epoch 11/100
212/212 ━━━━━━━━━━━━━━━━━━━━

In [7]:
# ============================================================
# EEGNET FIJO: 5 folds fijos + 10 semillas
# 10 seeds x 5 folds = 50 corridas limpias
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from collections import defaultdict
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from tensorflow.keras.optimizers import Adam

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

MODEL_NAME = "eegnet"

SEED_FOLDS = 123
TRAINING_SEEDS = [42, 123, 456, 789, 2024, 7, 99, 314, 2718, 5000]

N_FOLDS = 5
BATCH_SIZE = 64
EPOCHS_FINAL = 100

EXCLUDED_SUBJECTS = {"v56p"}

DATA_ROOT = "/kaggle/input/datasets/javierceron/tdha-data"

RESULTS_ROOT = "/kaggle/working/results_eegnet_10seeds"

PARTITIONS_DIR = os.path.join(RESULTS_ROOT, "partitions")

for d in [RESULTS_ROOT, PARTITIONS_DIR]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# SEED HELPER
# ============================================================

def set_all_seeds(seed, deterministic=True):
    tf.keras.backend.clear_session()
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    if deterministic:
        os.environ["TF_DETERMINISTIC_OPS"] = "1"
        try:
            tf.config.experimental.enable_op_determinism()
        except Exception:
            pass

# ============================================================
# HPs FIJOS EEGNET
# ============================================================

EEGNET_FIXED_MODEL_ARGS = {
    "nb_classes": 2,
    "dropoutRate": 0.5,
    "kernLength": 64,
    "F1": 8,
    "D": 2,
    "F2": 16,
    "norm_rate": 0.25,
    "dropoutType": "Dropout",
}

EEGNET_FIXED_COMPILE_ARGS = {
    "learning_rate": 1e-3,
}

# ============================================================
# DATASET BUILDER EEGNET: softmax -> one-hot
# ============================================================

def make_eegnet_ds_from_indices(X, y, idxs, training, batch_size, seed):
    x = X[idxs].astype(np.float32)
    y_bin = y[idxs].astype(np.int32)
    y_oh = tf.keras.utils.to_categorical(y_bin, num_classes=2).astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((x, y_oh))

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# ============================================================
# EVALUACIÓN A NIVEL DE SUJETO PARA SOFTMAX
# ============================================================

def evaluate_subject_level_softmax(model, test_ds_xyg, threshold=0.5):
    subject_preds = defaultdict(list)
    subject_probs = defaultdict(list)
    subject_true = {}

    for xb, yb, sb in test_ds_xyg:
        raw_pred = model(xb, training=False).numpy()

        if raw_pred.ndim != 2 or raw_pred.shape[1] != 2:
            raise ValueError(f"Se esperaba salida softmax (B,2), pero llegó {raw_pred.shape}")

        prob = raw_pred[:, 1].astype(np.float32)
        pred = (prob >= threshold).astype(int)

        y_np = yb.numpy().astype(int)
        s_np = sb.numpy()

        for name, p, pr, yt in zip(s_np, pred, prob, y_np):
            name = name.decode("utf-8") if isinstance(name, bytes) else str(name)
            subject_preds[name].append(int(p))
            subject_probs[name].append(float(pr))
            subject_true[name] = int(yt)

    y_true, y_pred, y_prob = [], [], []

    for subj in sorted(subject_preds.keys()):
        y_true.append(subject_true[subj])
        y_pred.append(int(np.bincount(subject_preds[subj]).argmax()))
        y_prob.append(float(np.mean(subject_probs[subj])))

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    if len(np.unique(y_true)) < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(y_true, y_prob)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "accuracy": float(acc),
        "balanced_acc": float(bacc),
        "f1": float(f1),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "cm": cm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

# ============================================================
# DATASET
# ============================================================

adhd_dir = os.path.join(DATA_ROOT, "ADHD")
control_dir = os.path.join(DATA_ROOT, "Control")

dataset_tf = EEGDataset_ADHD_TF(
    adhd_dir=adhd_dir,
    control_dir=control_dir,
    lowcut=0.5,
    highcut=60.0,
    notch=50.0,
    window=2.0,
    overlap=0.5,
    default_fs=128
)

n_before = len(dataset_tf.samples)

dataset_tf.samples = [
    sample for sample in dataset_tf.samples
    if os.path.splitext(sample[0])[0] not in EXCLUDED_SUBJECTS
]

n_after = len(dataset_tf.samples)

print(f"Excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
print(f"Subjects removed: {n_before - n_after}")
print(f"Remaining subjects: {n_after}")

if len(dataset_tf.samples) == 0:
    raise ValueError("No subjects left after exclusion.")

X, y, groups = build_epoch_arrays(dataset_tf)

N_CH, N_T = int(X.shape[1]), int(X.shape[2])

print("Epoch arrays:")
print("  X:", X.shape, X.dtype)
print("  y:", y.shape, "labels:", np.unique(y))
print("  groups:", groups.shape, "n_subjects:", len(np.unique(groups.astype(str))))
print("Model input shape (C,T):", (N_CH, N_T))

subj_names, subj_labels = build_subject_table_from_epochs(groups, y)

print("Subject table:")
print("  n_subjects:", len(subj_names))
print("  Control:", int(np.sum(subj_labels == 0)), "| ADHD:", int(np.sum(subj_labels == 1)))

# ============================================================
# PARTICIONES FIJAS
# ============================================================

outer = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED_FOLDS
)

X_dummy = np.zeros((len(subj_names), 1), dtype=np.float32)

FIXED_SPLITS = []
partitions_rows = []

for fold_id, (trainval_subj_idx, test_subj_idx) in enumerate(
    outer.split(X_dummy, subj_labels, groups=subj_names)
):
    print(f"\n================ FIXED FOLD {fold_id+1}/{N_FOLDS} ================")

    inner = StratifiedGroupKFold(
        n_splits=4,
        shuffle=True,
        random_state=SEED_FOLDS + fold_id
    )

    inner_X_dummy = np.zeros((len(trainval_subj_idx), 1), dtype=np.float32)
    inner_y = subj_labels[trainval_subj_idx]
    inner_groups = subj_names[trainval_subj_idx]

    tr_rel, val_rel = next(inner.split(inner_X_dummy, inner_y, groups=inner_groups))

    train_subj_idx = trainval_subj_idx[tr_rel]
    val_subj_idx = trainval_subj_idx[val_rel]

    tr_names = subj_names[train_subj_idx]
    va_names = subj_names[val_subj_idx]
    te_names = subj_names[test_subj_idx]

    tr_labels = subj_labels[train_subj_idx]
    va_labels = subj_labels[val_subj_idx]
    te_labels = subj_labels[test_subj_idx]

    if len(np.unique(tr_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: train set does not contain both classes.")
    if len(np.unique(va_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: validation set does not contain both classes.")
    if len(np.unique(te_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: test set does not contain both classes.")

    tr_idx = epoch_indices_from_subjects(groups, tr_names)
    val_idx = epoch_indices_from_subjects(groups, va_names)
    te_idx = epoch_indices_from_subjects(groups, te_names)

    trainval_names = np.concatenate([tr_names, va_names])
    trainval_idx = epoch_indices_from_subjects(groups, trainval_names)

    FIXED_SPLITS.append({
        "fold_id": fold_id,
        "tr_names": tr_names,
        "va_names": va_names,
        "te_names": te_names,
        "tr_idx": tr_idx,
        "val_idx": val_idx,
        "te_idx": te_idx,
        "trainval_idx": trainval_idx,
    })

    partitions_rows.append({
        "fold": fold_id,
        "train_subjects": list(tr_names),
        "val_subjects": list(va_names),
        "test_subjects": list(te_names),
        "n_train_subjects": len(tr_names),
        "n_val_subjects": len(va_names),
        "n_test_subjects": len(te_names),
        "n_train_epochs": len(tr_idx),
        "n_val_epochs": len(val_idx),
        "n_test_epochs": len(te_idx),
    })

df_partitions = pd.DataFrame(partitions_rows)
df_partitions.to_csv(
    os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"),
    index=False
)

print("\nParticiones fijas guardadas en:")
print(os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"))

# ============================================================
# CORRER UNA SEED
# ============================================================

def run_eegnet_one_seed(training_seed, save_artifacts=True):
    print(f"\n{'='*90}")
    print(f"RUN EEGNET | training_seed = {training_seed}")
    print(f"{'='*90}")

    seed_root = os.path.join(RESULTS_ROOT, f"seed_{training_seed}")
    curves_dir = os.path.join(seed_root, "training_curves")
    models_dir = os.path.join(seed_root, "models")
    cm_dir = os.path.join(seed_root, "cm")
    weights_dir = os.path.join(seed_root, "weights")

    for d in [seed_root, curves_dir, models_dir, cm_dir, weights_dir]:
        os.makedirs(d, exist_ok=True)

    fold_results = []
    fold_histories = {}
    fold_cms = {}
    super_cm = np.zeros((2, 2), dtype=int)

    for split in FIXED_SPLITS:
        fold_id = split["fold_id"]

        print(f"\n---------------- Fold {fold_id+1}/{N_FOLDS} | seed={training_seed} ----------------")

        fold_seed = int(training_seed * 100 + fold_id)

        trainval_idx = split["trainval_idx"]
        te_idx = split["te_idx"]

        trainval_ds_eegnet = make_eegnet_ds_from_indices(
            X, y, trainval_idx,
            training=True,
            batch_size=BATCH_SIZE,
            seed=fold_seed
        )

        test_ds_xyg = make_ds_from_indices(
            X, y, groups, te_idx,
            training=False,
            with_groups=True,
            batch_size=BATCH_SIZE,
            seed=fold_seed
        )

        set_all_seeds(fold_seed)

        model_args = dict(EEGNET_FIXED_MODEL_ARGS)
        model_args["Chans"] = N_CH
        model_args["Samples"] = N_T

        model = EEGNet(
            nb_classes=model_args["nb_classes"],
            Chans=model_args["Chans"],
            Samples=model_args["Samples"],
            dropoutRate=model_args["dropoutRate"],
            kernLength=model_args["kernLength"],
            F1=model_args["F1"],
            D=model_args["D"],
            F2=model_args["F2"],
            norm_rate=model_args["norm_rate"],
            dropoutType=model_args["dropoutType"],
        )

        opt = Adam(learning_rate=EEGNET_FIXED_COMPILE_ARGS["learning_rate"])

        model.compile(
            optimizer=opt,
            loss="categorical_crossentropy",
            metrics=[
                tf.keras.metrics.CategoricalAccuracy(name="acc"),
                tf.keras.metrics.AUC(name="auc"),
            ],
        )

        hist = model.fit(
            trainval_ds_eegnet,
            epochs=EPOCHS_FINAL,
            verbose=1,
        )

        fold_histories[fold_id] = hist.history

        if save_artifacts:
            model.save(os.path.join(models_dir, f"fold_{fold_id}.keras"))
            model.save_weights(os.path.join(weights_dir, f"fold_{fold_id}.weights.h5"))

            plot_fold_training_curve(
                hist.history,
                fold_id,
                save_path=os.path.join(curves_dir, f"fold_{fold_id}_curves.png"),
                metrics=("loss", "acc", "auc"),
            )

        fold_eval = evaluate_subject_level_softmax(
            model,
            test_ds_xyg,
            threshold=0.5
        )

        cm = fold_eval["cm"]
        super_cm += cm
        fold_cms[fold_id] = cm

        if save_artifacts:
            plot_fold_confusion_matrix(
                cm,
                fold_id,
                save_path=os.path.join(cm_dir, f"fold_{fold_id}_cm.png"),
            )

        row = {
            "seed": training_seed,
            "fold": fold_id,
            "accuracy": fold_eval["accuracy"],
            "balanced_acc": fold_eval["balanced_acc"],
            "f1": fold_eval["f1"],
            "auc": fold_eval["auc"],
            "model_args": dict(model_args),
            "compile_args": dict(EEGNET_FIXED_COMPILE_ARGS),
        }

        fold_results.append(row)

        print(
            f"Fold {fold_id} | "
            f"Acc={row['accuracy']:.3f} | "
            f"BAcc={row['balanced_acc']:.3f} | "
            f"F1={row['f1']:.3f} | "
            f"AUC={row['auc']:.3f}"
        )

        print("Confusion matrix:\n", cm)

        del model
        gc.collect()
        tf.keras.backend.clear_session()

    if save_artifacts:
        plot_all_training_curves(
            fold_histories,
            curves_dir,
            metrics=("loss", "acc", "auc"),
        )

        plot_all_confusion_matrices(
            fold_cms,
            super_cm,
            cm_dir
        )

    df_seed = pd.DataFrame(fold_results)

    seed_summary = {
        "seed": training_seed,
        "mean_accuracy": float(df_seed["accuracy"].mean()),
        "std_accuracy": float(df_seed["accuracy"].std(ddof=1)),
        "mean_balanced_acc": float(df_seed["balanced_acc"].mean()),
        "std_balanced_acc": float(df_seed["balanced_acc"].std(ddof=1)),
        "mean_f1": float(df_seed["f1"].mean()),
        "std_f1": float(df_seed["f1"].std(ddof=1)),
        "mean_auc": float(df_seed["auc"].mean()),
        "std_auc": float(df_seed["auc"].std(ddof=1)),
    }

    df_seed.to_csv(os.path.join(seed_root, "fold_results.csv"), index=False)
    pd.DataFrame([seed_summary]).to_csv(
        os.path.join(seed_root, "seed_summary.csv"),
        index=False
    )

    print("\nResumen seed:")
    print(pd.DataFrame([seed_summary]).T)

    return df_seed, seed_summary

# ============================================================
# CORRER LAS 10 SEMILLAS
# ============================================================

all_fold_results = []
all_seed_summaries = []

for training_seed in TRAINING_SEEDS:
    df_seed, seed_summary = run_eegnet_one_seed(
        training_seed=training_seed,
        save_artifacts=True
    )

    all_fold_results.append(df_seed)
    all_seed_summaries.append(seed_summary)

df_all_folds = pd.concat(all_fold_results, axis=0, ignore_index=True)
df_all_seeds = pd.DataFrame(all_seed_summaries)

df_all_folds.to_csv(
    os.path.join(RESULTS_ROOT, "all_fold_results.csv"),
    index=False
)

df_all_seeds.to_csv(
    os.path.join(RESULTS_ROOT, "all_seed_summaries.csv"),
    index=False
)

print("\nRESULTADOS POR SEED")
print(df_all_seeds)

print("\nRESUMEN GLOBAL")
print(
    df_all_seeds[
        ["mean_accuracy", "mean_balanced_acc", "mean_f1", "mean_auc"]
    ].agg(["mean", "std"])
)

print("\nArchivos principales guardados en:")
print(" -", os.path.join(RESULTS_ROOT, "all_fold_results.csv"))
print(" -", os.path.join(RESULTS_ROOT, "all_seed_summaries.csv"))
print(" -", os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"))

Excluded subjects: ['v56p']
Subjects removed: 1
Remaining subjects: 120
Epoch arrays:
  X: (16640, 19, 256) float32
  y: (16640,) labels: [0 1]
  groups: (16640,) n_subjects: 120
Model input shape (C,T): (19, 256)
Subject table:
  n_subjects: 120
  Control: 59 | ADHD: 61

================ FIXED FOLD 1/5 ================

================ FIXED FOLD 2/5 ================

================ FIXED FOLD 3/5 ================

================ FIXED FOLD 4/5 ================

================ FIXED FOLD 5/5 ================

Particiones fijas guardadas en:
/kaggle/working/results_eegnet_10seeds/partitions/fixed_partitions_summary.csv

RUN EEGNET | training_seed = 42

---------------- Fold 1/5 | seed=42 ----------------
Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1777001969.514105      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


204/204 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - acc: 0.5730 - auc: 0.5911 - loss: 0.6824
Epoch 2/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.6795 - auc: 0.7380 - loss: 0.6054
Epoch 3/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7424 - auc: 0.8076 - loss: 0.5401
Epoch 4/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7813 - auc: 0.8571 - loss: 0.4746
Epoch 5/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8108 - auc: 0.8908 - loss: 0.4187
Epoch 6/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8214 - auc: 0.9026 - loss: 0.3954
Epoch 7/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8410 - auc: 0.9199 - loss: 0.3619
Epoch 8/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8498 - auc: 0.9267 - loss: 0.3459
Epoch 9/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8639 - auc: 0.9360 - loss: 0.3248
Epoch 10/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8673 - auc: 0.9411 - loss: 0.3121
Epoch 11/100
204/204 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1777002204.333595      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


203/203 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - acc: 0.5552 - auc: 0.5698 - loss: 0.6907
Epoch 2/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - acc: 0.6632 - auc: 0.7189 - loss: 0.6191
Epoch 3/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7499 - auc: 0.8225 - loss: 0.5209
Epoch 4/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.7855 - auc: 0.8612 - loss: 0.4677
Epoch 5/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8067 - auc: 0.8887 - loss: 0.4252
Epoch 6/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8275 - auc: 0.9009 - loss: 0.4001
Epoch 7/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8332 - auc: 0.9097 - loss: 0.3837
Epoch 8/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8440 - auc: 0.9212 - loss: 0.3606
Epoch 9/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8550 - auc: 0.9300 - loss: 0.3406
Epoch 10/100
203/203 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - acc: 0.8587 - auc: 0.9339 - loss: 0.3310
Epoch 11/100
203/203 ━━━━━━━━━━━━━━━━━━━━

...

214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - acc: 0.5766 - auc: 0.5919 - loss: 0.6815
Epoch 2/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.6588 - auc: 0.7097 - loss: 0.6225
Epoch 3/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.7483 - auc: 0.8210 - loss: 0.5244
Epoch 4/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.7866 - auc: 0.8615 - loss: 0.4687
Epoch 5/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8094 - auc: 0.8881 - loss: 0.4244
Epoch 6/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8314 - auc: 0.9108 - loss: 0.3815
Epoch 7/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8370 - auc: 0.9162 - loss: 0.3704
Epoch 8/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8550 - auc: 0.9282 - loss: 0.3447
Epoch 9/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8464 - auc: 0.9263 - loss: 0.3480
Epoch 10/100
214/214 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8552 - auc: 0.9315 - loss: 0.3349
Epoch 11/100
214/214 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1777013365.141570      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


210/210 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - acc: 0.5447 - auc: 0.5619 - loss: 0.6886
Epoch 2/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.7137 - auc: 0.7787 - loss: 0.5690
Epoch 3/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.7797 - auc: 0.8601 - loss: 0.4678
Epoch 4/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8079 - auc: 0.8859 - loss: 0.4254
Epoch 5/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8288 - auc: 0.9065 - loss: 0.3911
Epoch 6/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8363 - auc: 0.9135 - loss: 0.3776
Epoch 7/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8478 - auc: 0.9257 - loss: 0.3497
Epoch 8/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8524 - auc: 0.9317 - loss: 0.3365
Epoch 9/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8707 - auc: 0.9406 - loss: 0.3143
Epoch 10/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8716 - auc: 0.9436 - loss: 0.3061
Epoch 11/100
210/210 ━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1777013597.215062      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - acc: 0.5578 - auc: 0.5751 - loss: 0.6871
Epoch 2/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.6760 - auc: 0.7316 - loss: 0.6093
Epoch 3/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.7356 - auc: 0.8031 - loss: 0.5420
Epoch 4/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.7698 - auc: 0.8463 - loss: 0.4912
Epoch 5/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.7994 - auc: 0.8797 - loss: 0.4388
Epoch 6/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8086 - auc: 0.8936 - loss: 0.4117
Epoch 7/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8348 - auc: 0.9130 - loss: 0.3752
Epoch 8/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8511 - auc: 0.9287 - loss: 0.3421
Epoch 9/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8527 - auc: 0.9310 - loss: 0.3364
Epoch 10/100
212/212 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - acc: 0.8622 - auc: 0.9344 - loss: 0.3281
Epoch 11/100
212/212 ━━━━━━━━━━━━━━━━━━━━